# Mapeamento OCDS

Este script tem como objetivo facilitar o processo de debugging do mapeamento OCDS. Ele lê um arquivo JSON contendo os dados mapeados e exibe as informações de forma organizada, permitindo uma análise mais eficiente.

## Setup de dados

In [3]:
#----------------------------------
# Importando bibliotecas
#----------------------------------
import pandas as pd

#----------------------------------
# DIRETORIO
#----------------------------------
base_dir = "C:/Users/gabri/OneDrive/Área de Trabalho/joao/TB/cesta-de-precos-pncp"

#--------------------------
# INPUTS
#--------------------------
INPUT_DATASET_MEDICAMENTOS = "tasks/mapeamento-ocds/input/dicionario-ocds.csv"
ITENS = r"coleta\data-package\2026\3 - Março\QUINZENA-1\DATA\itens-medicamentos.csv"

medicamentos_ocds = pd.read_csv(f"{base_dir}/{INPUT_DATASET_MEDICAMENTOS}")
itens = pd.read_csv(f"{base_dir}/{ITENS}")

### Datasets

- `itens`: contratações de medicamentos no PNCP, resultado das coletas quinzenais. Esse é o dataset que alimenta a base de dados do Medicamentos Transparentes.
- `medicamentos_ocds`: mapeamento dos itens de medicamentos do CATMAT para o formato OCDS. Esse dataset é o resultado do processo de mapeamento da extensão de medicamentos.

## Unificação e inspeção inicial

Faz a leitura e o left-join de `itens` e `medicamentos_ocds`

In [ ]:
itens = itens.merge(medicamentos_ocds, how="left", on="codigo_br")
itens.head()

,numeroItem,endpoint,descricao,materialOuServico,materialOuServicoNome,valorUnitarioEstimado,valorTotal,quantidade,unidadeMedida,orcamentoSigiloso,...,dosageForm,administrationRoute,activeIngredients.strengthValue,activeIngredients.strengthUnit,immediateContainer,origem_activeIngredients,origem_dosageForm,origem_administrationRoute,origem_strength,origem_immediateContainer
0,2,https://pncp.gov.br/api/pncp/v1/orgaos/0859712...,Polihexanida,M,Material,39.31,1886.88,48.0,"Frasco 40,00 ML",False,...,['Solução Aquosa'],NaN,"['0,1%', '0,1%']",NaN,NaN,['Composição'],['Forma Farmacêutica'],NaN,"['Concentração', 'Concentração']",NaN
1,3,https://pncp.gov.br/api/pncp/v1/orgaos/0859712...,Polihexanida,M,Material,49.12,2357.76,48.0,"Frasco 30,00 ML",False,...,['Solução Aquosa'],NaN,"['0,1%', '0,1%']",NaN,NaN,['Composição'],['Forma Farmacêutica'],NaN,"['Concentração', 'Concentração']",NaN
2,4,https://pncp.gov.br/api/pncp/v1/orgaos/0859712...,Polihexanida,M,Material,37.73,2716.56,72.0,"Frasco 100,00 ML",False,...,['Solução Aquosa'],NaN,"['0,1%', '0,1%']",NaN,NaN,['Composição'],['Forma Farmacêutica'],NaN,"['Concentração', 'Concentração']",NaN
3,1,https://pncp.gov.br/api/pncp/v1/orgaos/0859712...,Carbonato de cálcio,M,Material,4.90,14112.00,2880.0,Cápsula,False,...,NaN,NaN,['500'],['MG'],NaN,NaN,NaN,NaN,['Dosagem'],NaN
4,2,https://pncp.gov.br/api/pncp/v1/orgaos/0859712...,Citrato de cálcio,M,Material,5.29,19044.00,3600.0,Comprimido,False,...,NaN,NaN,"['250', '2,5']","['MG', 'MCG']",NaN,['Composição'],NaN,NaN,"['Concentração', 'Concentração']",NaN


### Selecionar uma contratação

In [5]:
ITEM1 = itens.sample(1, random_state=0)
ITEM1

,numeroItem,endpoint,descricao,materialOuServico,materialOuServicoNome,valorUnitarioEstimado,valorTotal,quantidade,unidadeMedida,orcamentoSigiloso,...,dosageForm,administrationRoute,activeIngredients.strengthValue,activeIngredients.strengthUnit,immediateContainer,origem_activeIngredients,origem_dosageForm,origem_administrationRoute,origem_strength,origem_immediateContainer
2846,5,https://pncp.gov.br/api/pncp/v1/orgaos/0859712...,Dexpantenol,M,Material,46.68,2240.64,48.0,"Bisnaga 15,00 G",False,...,NaN,['Gel Oftálmico'],['50'],['MG/G'],NaN,NaN,NaN,['Forma Farmacêutica'],['Concentração'],NaN


## Exemplo de Release package OCDS para ITEM1

Este trecho adapta a montagem de `mapeamento-pncp-ocds.py` para o exemplo manual do notebook usando `ITEM1`, que é a amostra de `itens` unificada com `dicionario-ocds.csv`.


In [6]:
from datetime import datetime, timezone
from pathlib import Path
import json
import zipfile
import pandas as pd

DATA_DIR = Path(base_dir) / Path(ITENS).parent
CONTRATACOES = DATA_DIR / "contratacoes.csv"
RESULTADOS = DATA_DIR / "itens-medicamentos-resultados.csv"

contratacoes_item1 = pd.read_csv(CONTRATACOES)
resultados_item1 = pd.read_csv(RESULTADOS)


def endpoint_para_numero_controle(endpoint):
    partes = endpoint.strip("/").split("/")
    cnpj = partes[-5]
    ano = partes[-3]
    sequencial = partes[-2]
    return f"{cnpj}-1-{int(sequencial):06d}/{ano}"


if "data.numeroControlePNCP" not in ITEM1.columns:
    ITEM1 = ITEM1.copy()
    ITEM1["data.numeroControlePNCP"] = ITEM1["endpoint"].apply(endpoint_para_numero_controle)


def valor_preenchido(valor):
    if isinstance(valor, (list, tuple, dict)):
        return True
    return pd.notna(valor)


def codigo_ocds(valor):
    if not valor_preenchido(valor):
        return None
    if isinstance(valor, float) and valor.is_integer():
        return str(int(valor))
    return str(valor)


def valor_json(valor):
    if isinstance(valor, dict):
        return {k: valor_json(v) for k, v in valor.items() if valor_json(v) is not None}
    if isinstance(valor, list):
        return [valor_json(v) for v in valor]
    if not valor_preenchido(valor):
        return None
    if hasattr(valor, "item"):
        return valor.item()
    return valor


def to_ocds_dt(data, field=None):
    try:
        dt = datetime.fromisoformat(data).astimezone(timezone.utc)
        return dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    except Exception as e:
        field_str = f" campo='{field}'" if field else ""
        print(f"Erro ao converter data:{field_str} valor='{data}' tipo={type(data)} erro={e}")
        return None


def duracao_em_dias(data_inicio, data_fim):
    try:
        inicio = datetime.fromisoformat(data_inicio)
        fim = datetime.fromisoformat(data_fim)
        return (fim - inicio).days
    except Exception as e:
        print(f"Erro ao calcular duração: {e} ->> {data_inicio}: {type(data_inicio)} || ->> {data_fim}: {type(data_fim)}")
        return None


def montar_release_package_item1(ITEM1, contratacoes, resultados):
    numero_controle = ITEM1["data.numeroControlePNCP"].dropna().iloc[0]
    contratacoes_filtradas = contratacoes[contratacoes["data.numeroControlePNCP"] == numero_controle].copy()
    itens_filtrados = ITEM1[ITEM1["data.numeroControlePNCP"] == numero_controle].copy()

    if contratacoes_filtradas.empty:
        raise ValueError(f"Contratação {numero_controle} não encontrada em contratacoes.csv")

    releases = []

    award_criteria = {
        "1": "priceOnly",
        "2": "priceOnly",
        "3": "qualityOnly",
        "4": "ratedCriteria",
        "5": "priceOnly",
        "6": "costOnly",
        "7": None,
        "8": "qualityOnly",
    }

    tipo_pessoa = {
        "PF": "Pessoa Física",
        "PJ": "Pessoa Jurídica",
        "PE": "Pessoa Estrangeira",
    }

    porte_fornecedor = {
        "ME": "micro",
        "EPP": "small",
        "Demais": "large",
        "Não se aplica": "self-employed",
        "Não Informado": None,
    }

    poder_id = {
        "E": "Executivo",
        "L": "Legislativo",
        "J": "Judiciário",
        "N": "Não se aplica",
    }

    esfera_id = {
        "F": "Federal",
        "E": "Estadual",
        "M": "Municipal",
        "D": "Distrital",
        "N": "Não se aplica",
    }

    tipo_instrumento_convocatorio = {
        "1": "open",
        "2": "limited",
        "3": "direct",
        "4": "selective",
    }

    modalidade_contratacao = {
        "1": "electronicAuction",
        "2": None,
        "3": None,
        "4": "electronicSubmission",
        "5": "written",
        "6": "electronicSubmission",
        "7": "written",
        "8": None,
        "9": None,
        "10": None,
        "11": None,
        "12": None,
        "13": "written",
        "14": None,
    }

    import ast
    import re


    def parse_lista(valor):
        if not valor_preenchido(valor):
            return []

        texto = str(valor).strip()

        try:
            valor_parseado = ast.literal_eval(texto)
        except (SyntaxError, ValueError):
            return [texto] if texto else []

        if isinstance(valor_parseado, list):
            return [str(v).strip() for v in valor_parseado if valor_preenchido(v)]

        return [str(valor_parseado).strip()] if valor_preenchido(valor_parseado) else []


    def primeiro_valor(valor):
        valores = parse_lista(valor)
        return valores[0] if valores else None


    def normalizar_strength_value(valor, unidade=None):
        if not valor_preenchido(valor):
            return None

        texto = str(valor).strip()
        eh_percentual = "%" in texto or str(unidade or "").strip().startswith("%")
        texto_numerico = texto.replace("%", "").strip()
        texto_normalizado = texto_numerico.replace(".", "").replace(",", ".")

        try:
            numero = float(texto_normalizado)
        except ValueError:
            return None

        if eh_percentual:
            numero = round(numero / 100, 12)

        return int(numero) if numero.is_integer() else numero


    def montar_strength(valor, unidade):
        strength = {}

        valor_normalizado = normalizar_strength_value(valor, unidade)
        if valor_normalizado is not None:
            strength["value"] = valor_normalizado

        if valor_preenchido(unidade):
            strength["unit"] = {
                "scheme": "UNCEFACT",
                "id": str(unidade).strip(),
            }

        return strength


    def separar_nome_active_ingredient(nome):
        if not valor_preenchido(nome):
            return []

        texto = str(nome).strip()

        if re.search(r"(?i)\bassoc|(?:\bc\s*/)", texto):
            texto = re.sub(
                r"(?i)^\s*(?:associad[oa]s?|assoc\.?)\s*(?:com|[àa]|ao)?\s*",
                "",
                texto,
            )
            texto = re.sub(r"(?i)^\s*c\s*/\s*", "", texto)

            partes = [
                parte.strip()
                for parte in re.split(r"\s*(?:;|\+|,|\b[Ee]\b)\s*", texto)
                if parte.strip()
            ]
            if partes:
                return partes

        return [texto]


    def montar_active_ingredients(item):
        nomes = []
        for nome in parse_lista(item.get("activeIngredients.name")):
            for parte in re.split(r"\s*;\s*", nome):
                nomes.extend(separar_nome_active_ingredient(parte))

        nome_pdm = str(item.get("nome_pdm")).strip() if valor_preenchido(item.get("nome_pdm")) else ""
        valores_strength = parse_lista(item.get("activeIngredients.strengthValue"))
        unidades_strength = parse_lista(item.get("activeIngredients.strengthUnit"))

        if nome_pdm and (not nomes or len(valores_strength) > len(nomes)):
            nomes = [nome_pdm, *nomes]

        ingredientes = []
        total = max(len(nomes), len(valores_strength), len(unidades_strength))

        for idx in range(total):
            ingrediente = {}

            if idx < len(nomes):
                ingrediente["name"] = nomes[idx]
            elif nomes:
                ingrediente["name"] = nomes[-1]

            valor = valores_strength[idx] if idx < len(valores_strength) else None
            unidade = unidades_strength[idx] if idx < len(unidades_strength) else None
            strength = montar_strength(valor, unidade)

            if strength:
                ingrediente["strength"] = strength

            if ingrediente:
                ingredientes.append(ingrediente)

        if nome_pdm and (
            not ingredientes
            or ingredientes[0].get("name", "").casefold() != nome_pdm.casefold()
        ):
            ingredientes.insert(0, {"name": nome_pdm})

        return ingredientes

    for _, row in contratacoes_filtradas.iterrows():
        ocid = (
            str(row["data.orgaoEntidade.cnpj"])
            + "_"
            + str(row["data.anoCompra"])
            + "_"
            + str(row["data.sequencialCompra"])
        )

        itens_rel = itens_filtrados[itens_filtrados["data.numeroControlePNCP"] == row["data.numeroControlePNCP"]]
        items = []
        items_by_id = {}
        lots = []

        for _, item in itens_rel.iterrows():
            item_id = codigo_ocds(item["numeroItem"])
            unit = {
                "value": {
                    "amount": valor_json(item["valorUnitarioEstimado"]),
                    "currency": "BRL",
                }
            }

            if valor_preenchido(item["unidadeMedida"]):
                unit["name"] = item["unidadeMedida"]

            i = {
                "id": item_id,
                "description": item["descricao"],
                "classification": {
                    "scheme": "CATMAT",
                    "id": codigo_ocds(item["codigo_br"]),
                    "description": item["desc_item"],
                    "uri": (
                        "https://cnbs.estaleiro.serpro.gov.br/cnbs-api/material/v1/"
                        "recuperaDadosItemMaterialPorCodigo?codigo_item_material="
                        + codigo_ocds(item["codigo_br"])
                    ),
                },
                **({"dosageForm": primeiro_valor(item["dosageForm"])} if valor_preenchido(item["dosageForm"]) else {}),
                **({"administrationRoute": primeiro_valor(item["administrationRoute"])} if valor_preenchido(item["administrationRoute"]) else {}),
                **({"activeIngredients": montar_active_ingredients(item)} if montar_active_ingredients(item) else {}),
                "quantity": int(item["quantidade"]),
                "unit": unit,
                "relatedLot": "lot-" + item_id,
            }

            items.append(i)
            items_by_id[item_id] = i

            criterio = award_criteria.get(codigo_ocds(item["criterioJulgamentoId"]))
            beneficio_nome = item["tipoBeneficioNome"] if valor_preenchido(item["tipoBeneficioNome"]) else None

            lot = {
                "id": "lot-" + item_id,
                **({"statusDetailsId": codigo_ocds(item["situacaoCompraItem"])} if valor_preenchido(item["situacaoCompraItem"]) else {}),
                **({"statusDetails": item["situacaoCompraItemNome"]} if valor_preenchido(item["situacaoCompraItemNome"]) else {}),
                "value": {
                    "amount": valor_json(item["valorTotal"]),
                    "currency": "BRL",
                },
                "lotsDetails": {
                    **({"confidentialBudget": valor_json(item["orcamentoSigiloso"])} if valor_preenchido(item["orcamentoSigiloso"]) and item["orcamentoSigiloso"] != False else {}),
                    **({"asset": valor_json(item["patrimonio"])} if valor_preenchido(item["patrimonio"]) else {}),
                    **({"realEstateRegistrationCode": valor_json(item["codigoRegistroImobiliario"])} if valor_preenchido(item["codigoRegistroImobiliario"]) else {}),
                    **({"standardPreferenceMarginApplicability": valor_json(item["aplicabilidadeMargemPreferenciaNormal"])} if valor_preenchido(item["aplicabilidadeMargemPreferenciaNormal"]) and item["aplicabilidadeMargemPreferenciaNormal"] != False else {}),
                    **({"standardPreferenceMarginPercentage": valor_json(item["percentualMargemPreferenciaNormal"])} if valor_preenchido(item["percentualMargemPreferenciaNormal"]) else {}),
                    **({"additionalPreferenceMarginApplicability": valor_json(item["aplicabilidadeMargemPreferenciaAdicional"])} if valor_preenchido(item["aplicabilidadeMargemPreferenciaAdicional"]) and item["aplicabilidadeMargemPreferenciaAdicional"] != False else {}),
                    **({"additionalPreferenceMarginPercentage": valor_json(item["percentualMargemPreferenciaAdicional"])} if valor_preenchido(item["percentualMargemPreferenciaAdicional"]) else {}),
                    **({
                        **({"benefitType": codigo_ocds(item["tipoBeneficio"])} if valor_preenchido(item["tipoBeneficio"]) else {}),
                        **({"benefitTypeName": beneficio_nome} if beneficio_nome else {}),
                    } if beneficio_nome not in [None, "Sem benefício", "Não se aplica"] else {}),
                    **({"ncmNbsCode": valor_json(item["ncmNbsCodigo"])} if valor_preenchido(item["ncmNbsCodigo"]) else {}),
                    **({"ncmNbsDescription": item["ncmNbsDescricao"]} if valor_preenchido(item["ncmNbsDescricao"]) else {}),
                    **({"catalogItemCategoryId": valor_json(item["categoriaItemCatalogo.id"])} if valor_preenchido(item["categoriaItemCatalogo.id"]) else {}),
                    **({"catalogItemCategoryName": item["categoriaItemCatalogo.nome"]} if valor_preenchido(item["categoriaItemCatalogo.nome"]) else {}),
                    **({"catalogItemCategoryDescription": item["categoriaItemCatalogo.descricao"]} if valor_preenchido(item["categoriaItemCatalogo.descricao"]) else {}),
                    **({"catalogItemCode": valor_json(item["catalogoCodigoItem"])} if valor_preenchido(item["catalogoCodigoItem"]) else {}),
                    **({"awardCriteria": criterio} if criterio else {}),
                    **({"awardCriteriaDetails": item["criterioJulgamentoNome"]} if valor_preenchido(item["criterioJulgamentoNome"]) else {}),
                },
            }

            if beneficio_nome == "Participação exclusiva para ME/EPP":
                lot["sustainability"] = [{"goal": "social.smeInclusion", "strategies": ["reservedParticipation"]}]
                lot["otherRequirements"] = {"reservedParticipation": ["sme"]}
            elif beneficio_nome == "Subcontratação para ME/EPP":
                lot["sustainability"] = [{"goal": "social.smeInclusion", "strategies": ["subcontracting"]}]
                lot["subcontractingTerms"] = {"description": "Subcontratação para ME/EPP"}
            elif beneficio_nome == "Cota reservada para ME/EPP":
                lot["sustainability"] = [{"goal": "social.smeInclusion", "strategies": ["reservedParticipationQuota"]}]

            lots.append(lot)

        res_rel = resultados[
            (resultados["numeroControlePNCPCompra"] == row["data.numeroControlePNCP"])
            & (resultados["numeroItem"].map(codigo_ocds).isin(items_by_id.keys()))
        ]
        awards = []
        lista_ni_fornecedor = []
        suppliers = []

        for idx, (_, res) in enumerate(res_rel.iterrows(), start=1):
            fornecedor_id = "BR-CNPJ-" + str(res["niFornecedor"])

            if str(res["niFornecedor"]) not in lista_ni_fornecedor:
                lista_ni_fornecedor.append(str(res["niFornecedor"]))
                fornecedor = {
                    "id": fornecedor_id,
                    "name": res["nomeRazaoSocialFornecedor"],
                    "identifier": {
                        "scheme": "BR-CNPJ",
                        "id": str(res["niFornecedor"]),
                        "legalName": res["nomeRazaoSocialFornecedor"],
                    },
                    "address": {"countryName": res["codigoPais"]},
                    "roles": ["supplier"],
                    "details": {
                        **({"scale": porte_fornecedor.get(res["porteFornecedorNome"])} if porte_fornecedor.get(res["porteFornecedorNome"]) else {}),
                        "classifications": [
                            {
                                "scheme": "BRA-TIPO-PESSOA",
                                "id": str(res["tipoPessoa"]),
                                "description": tipo_pessoa.get(res["tipoPessoa"], res["tipoPessoa"]),
                            },
                            {
                                "scheme": "ORDEM DE CLASSIFICACAO SRP",
                                **({"id": codigo_ocds(res["ordemClassificacaoSrp"])} if valor_preenchido(res["ordemClassificacaoSrp"]) else {}),
                            },
                        ],
                    },
                }

                if valor_preenchido(res["naturezaJuridicaId"]):
                    fornecedor["details"]["classifications"].append({
                        "scheme": "BRA-NATUREZA-JURIDICA",
                        "id": codigo_ocds(res["naturezaJuridicaId"]),
                        "description": str(res["naturezaJuridicaNome"]),
                    })

                suppliers.append(fornecedor)

            res_item_id = codigo_ocds(res["numeroItem"])
            item_ref = items_by_id[res_item_id]
            awards.append({
                "id": str(idx),
                "title": row["data.tipoInstrumentoConvocatorioNome"] + " - " + str(row["data.processo"]),
                "description": row["data.objetoCompra"],
                "date": to_ocds_dt(res["dataResultado"], field="dataResultado"),
                "hasSubcontracting": valor_json(res["indicadorSubcontratacao"]),
                "value": {
                    "amount": valor_json(res["valorTotalHomologado"]),
                    "currency": "BRL",
                },
                "suppliers": [{"id": fornecedor_id, "name": res["nomeRazaoSocialFornecedor"]}],
                "items": [{
                    "id": res_item_id,
                    "description": item_ref["description"],
                    "status": res["situacaoCompraItemResultadoNome"],
                    **({"statusDetails": res["motivoCancelamento"]} if valor_preenchido(res["motivoCancelamento"]) else {}),
                    "quantity": int(res["quantidadeHomologada"]),
                    "unit": {
                        **({"name": item_ref["unit"]["name"]} if "name" in item_ref["unit"] else {}),
                        "value": {
                            "amount": valor_json(res["valorUnitarioHomologado"]),
                            "currency": "BRL",
                        },
                    },
                    **({"deliveryLocations": [{"description": str(res["paisOrigemProdutoServico.nome"])}]} if valor_preenchido(res["paisOrigemProdutoServico.nome"]) else {}),
                    "relatedLot": "lot-" + res_item_id,
                }],
            })

        buyer_cnpj = row["data.orgaoSubRogado.cnpj"] if valor_preenchido(row["data.orgaoSubRogado.cnpj"]) else row["data.orgaoEntidade.cnpj"]
        buyer_name = row["data.orgaoSubRogado.razaoSocial"] if valor_preenchido(row["data.orgaoSubRogado.razaoSocial"]) else row["data.orgaoEntidade.razaoSocial"]
        buyer_esfera = row["data.orgaoSubRogado.esferaId"] if valor_preenchido(row["data.orgaoSubRogado.esferaId"]) else row["data.orgaoEntidade.esferaId"]
        buyer_poder = row["data.orgaoSubRogado.poderId"] if valor_preenchido(row["data.orgaoSubRogado.poderId"]) else row["data.orgaoEntidade.poderId"]

        release = {
            "ocid": "ocds-ye9ov3-" + ocid,
            "id": row["data.numeroControlePNCP"],
            "date": to_ocds_dt(row["data.dataPublicacaoPncp"], field="data.dataPublicacaoPncp"),
            "tag": ["tender", "award"],
            "initiationType": "tender",
            "buyer": {"id": "BR-CNPJ-" + str(buyer_cnpj), "name": buyer_name},
            "language": "pt",
            "parties": [],
        }

        buyer_party = {
            "id": "BR-CNPJ-" + str(buyer_cnpj),
            "name": buyer_name,
            "identifier": {"scheme": "BR-CNPJ", "id": str(buyer_cnpj), "legalName": buyer_name},
            "additionalIdentifiers": [{
                "id": str(row["data.unidadeSubRogada.codigoUnidade"]) if valor_preenchido(row["data.unidadeSubRogada.codigoUnidade"]) else str(row["data.unidadeOrgao.codigoUnidade"]),
                "legalName": row["data.unidadeSubRogada.nomeUnidade"] if valor_preenchido(row["data.unidadeSubRogada.nomeUnidade"]) else row["data.unidadeOrgao.nomeUnidade"],
            }],
            "address": {
                "region": row["data.unidadeSubRogada.ufNome"] if valor_preenchido(row["data.unidadeSubRogada.ufNome"]) else row["data.unidadeOrgao.ufNome"],
                "locality": row["data.unidadeSubRogada.municipioNome"] if valor_preenchido(row["data.unidadeSubRogada.municipioNome"]) else row["data.unidadeOrgao.municipioNome"],
            },
            "roles": ["buyer", "procuringEntity"],
            "details": {
                "classifications": [{
                    "scheme": "BRA-ESFERA",
                    "id": "BRA-ESFERA-" + str(buyer_esfera),
                    "description": esfera_id.get(buyer_esfera, buyer_esfera),
                }]
            },
        }

        if str(buyer_poder) != "N":
            buyer_party["details"]["classifications"].append({
                "scheme": "BRA-PODER",
                "id": "BRA-PODER-" + str(buyer_poder),
                "description": poder_id.get(buyer_poder, buyer_poder),
            })

        release["parties"].append(buyer_party)

        if valor_preenchido(row["data.orgaoSubRogado.cnpj"]):
            original_buyer = {
                "id": "BR-CNPJ-" + str(row["data.orgaoEntidade.cnpj"]),
                "name": row["data.orgaoEntidade.razaoSocial"],
                "identifier": {
                    "scheme": "BR-CNPJ",
                    "id": str(row["data.orgaoEntidade.cnpj"]),
                    "legalName": row["data.orgaoEntidade.razaoSocial"],
                },
                "additionalIdentifiers": [{
                    "id": str(row["data.unidadeOrgao.codigoUnidade"]),
                    "legalName": row["data.unidadeOrgao.nomeUnidade"],
                }],
                "address": {
                    "region": row["data.unidadeOrgao.ufNome"],
                    "locality": row["data.unidadeOrgao.municipioNome"],
                },
                "roles": ["originalBuyer"],
                "details": {
                    "classifications": [{
                        "scheme": "BRA-ESFERA",
                        "id": "BRA-ESFERA-" + str(row["data.orgaoEntidade.esferaId"]),
                        "description": esfera_id.get(row["data.orgaoEntidade.esferaId"], row["data.orgaoEntidade.esferaId"]),
                    }]
                },
            }

            if str(row["data.orgaoEntidade.poderId"]) != "N":
                original_buyer["details"]["classifications"].append({
                    "scheme": "BRA-PODER",
                    "id": "BRA-PODER-" + str(row["data.orgaoEntidade.poderId"]),
                    "description": poder_id.get(row["data.orgaoEntidade.poderId"], row["data.orgaoEntidade.poderId"]),
                })

            release["parties"].append(original_buyer)

        release["parties"] = release["parties"] + suppliers

        modalidade = modalidade_contratacao.get(codigo_ocds(row["data.modalidadeId"]))
        release["tender"] = {
            "id": str(row["data.processo"]),
            "title": row["data.tipoInstrumentoConvocatorioNome"] + " - " + str(row["data.processo"]),
            "description": str(row["data.objetoCompra"]),
            "procuringEntity": {
                "name": row["data.orgaoEntidade.razaoSocial"],
                "id": "BR-CNPJ-" + str(row["data.orgaoEntidade.cnpj"]),
            },
            "value": {"amount": valor_json(row["data.valorTotalEstimado"]), "currency": "BRL"},
            "procurementMethod": tipo_instrumento_convocatorio.get(codigo_ocds(row["data.tipoInstrumentoConvocatorioCodigo"])),
            "procurementMethodDetails": (
                "tipoInstrumentoConvocatorioCodigo: " + str(row["data.tipoInstrumentoConvocatorioCodigo"])
                + "; tipoInstrumentoConvocatorioNome: " + row["data.tipoInstrumentoConvocatorioNome"]
                + "; modalidadeId: " + str(row["data.modalidadeId"])
                + "; modalidadeNome: " + row["data.modalidadeNome"]
                + "; modoDisputaId: " + str(row["data.modoDisputaId"])
                + "; modoDisputaNome: " + row["data.modoDisputaNome"]
                + "; srp: " + str(row["data.srp"])
            ),
            "procurementMethodRationale": (
                "amparoLegalCodigo: " + str(row["data.amparoLegal.codigo"])
                + "; amparoLegalNome: " + str(row["data.amparoLegal.nome"])
                + "; amparoLegalDescricao: " + str(row["data.amparoLegal.descricao"])
            ),
            "mainProcurementCategory": "goods",
            **({"submissionMethod": [modalidade]} if modalidade else {}),
            "submissionMethodDetails": str(row["data.modalidadeNome"]),
            **({"tenderPeriod": {
                **({"startDate": to_ocds_dt(row["data.dataAberturaProposta"], field="data.dataAberturaProposta")} if valor_preenchido(row["data.dataAberturaProposta"]) else {}),
                **({"endDate": to_ocds_dt(row["data.dataEncerramentoProposta"], field="data.dataEncerramentoProposta")} if valor_preenchido(row["data.dataEncerramentoProposta"]) else {}),
                **({"maxExtentDate": to_ocds_dt(row["data.dataEncerramentoProposta"], field="data.dataEncerramentoProposta")} if valor_preenchido(row["data.dataEncerramentoProposta"]) else {}),
                **({"durationInDays": duracao_em_dias(row["data.dataAberturaProposta"], row["data.dataEncerramentoProposta"])} if valor_preenchido(row["data.dataAberturaProposta"]) and valor_preenchido(row["data.dataEncerramentoProposta"]) else {}),
            }} if valor_preenchido(row["data.dataAberturaProposta"]) or valor_preenchido(row["data.dataEncerramentoProposta"]) else {}),
            "items": items,
            "lots": lots,
        }

        if awards:
            release["awards"] = awards

        releases.append(valor_json(release))

    estado = str(contratacoes_filtradas.iloc[0]["data.unidadeOrgao.ufSigla"]).lower()
    mes_publicacao = Path(ITENS).parts[-4].split(" - ")[0]
    ano_publicacao = Path(ITENS).parts[-5]
    package_id = f"{estado}-{mes_publicacao}-{ano_publicacao}"

    return valor_json({
        "uri": f"https://medicamentos-transparentes-dados-abertos.s3.sa-east-1.amazonaws.com/{package_id}-json.zip",
        "publishedDate": to_ocds_dt(datetime.now().isoformat(), field="publisherDate"),
        "publisher": {
            "name": "Medicamentos Transparentes",
            "uri": "https://medicamentos.transparencia.org.br/",
        },
        "version": "1.1",
        "license": "https://creativecommons.org/licenses/by/4.0/",
        "extensions": [
            "https://raw.githubusercontent.com/open-contracting-extensions/ocds_partyDetails_scale_extension/master/extension.json",
            "https://raw.githubusercontent.com/open-contracting-extensions/ocds_lots_extension/v1.1.5/extension.json",
            "https://raw.githubusercontent.com/open-contracting-extensions/ocds_subcontracting_extension/master/extension.json",
            "https://raw.githubusercontent.com/open-contracting-extensions/ocds_location_extension/master/extension.json",
            "https://raw.githubusercontent.com/open-contracting-extensions/ocds_organizationClassification_extension/master/extension.json",
            "https://raw.githubusercontent.com/open-contracting-extensions/ocds_sustainability_extension/master/extension.json",
            "https://gitlab.com/dncp-opendata/ocds_statusdetails_extension/-/raw/master/extension.json",
        ],
        "releases": releases,
    })


release_package_item1 = montar_release_package_item1(ITEM1, contratacoes_item1, resultados_item1)
json_item1 = json.dumps(release_package_item1, ensure_ascii=False, indent=2)

package_id_item1 = Path(release_package_item1["uri"]).name.removesuffix("-json.zip")
ano_publicacao_item1 = Path(ITENS).parts[-5]
mes_publicacao_item1 = Path(ITENS).parts[-4].split(" - ")[0]

json_dir_item1 = Path(base_dir) / "tasks" / "mapeamento-ocds" / "output"
json_dir_item1.mkdir(parents=True, exist_ok=True)
json_file_item1 = json_dir_item1 / f"item1-{package_id_item1}.json"
json_file_item1.write_text(json_item1, encoding="utf-8")

print(f"Release package gerado com {len(release_package_item1['releases'])} release(s).")
print(f"JSON salvo em: {json_file_item1}")

print(json_item1)


Release package gerado com 1 release(s).
JSON salvo em: C:\Users\gabri\OneDrive\Área de Trabalho\joao\TB\cesta-de-precos-pncp\tasks\mapeamento-ocds\output\item1-pr-3-2026.json
{
  "uri": "https://medicamentos-transparentes-dados-abertos.s3.sa-east-1.amazonaws.com/pr-3-2026-json.zip",
  "publishedDate": "2026-06-07T21:48:31Z",
  "publisher": {
    "name": "Medicamentos Transparentes",
    "uri": "https://medicamentos.transparencia.org.br/"
  },
  "version": "1.1",
  "license": "https://creativecommons.org/licenses/by/4.0/",
  "extensions": [
    "https://raw.githubusercontent.com/open-contracting-extensions/ocds_partyDetails_scale_extension/master/extension.json",
    "https://raw.githubusercontent.com/open-contracting-extensions/ocds_lots_extension/v1.1.5/extension.json",
    "https://raw.githubusercontent.com/open-contracting-extensions/ocds_subcontracting_extension/master/extension.json",
    "https://raw.githubusercontent.com/open-contracting-extensions/ocds_location_extension/maste